## WESAD — ¿es utilizable la frecuencia respiratoria estimada desde la PPG de muñeca?
Paso de factibilidad antes de diseñar el estudio de estrés.
- **Referencia:** respiración del pecho (RespiBAN, 700 Hz): respiraciones detectadas en la señal.
- **Estimación:** PPG de muñeca (Empatica E4, 64 Hz) con la FFT congelada de BIDMC/CapnoBase,
  sin cambiar ningún parámetro.
- Resultados por **condición** (línea base, estrés, diversión, meditación): en estrés (TSST) hay habla.

In [ ]:
# w-1 — Imports, configuración congelada y rutas
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.signal import butter, decimate, detrend, filtfilt, find_peaks, resample_poly

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CFG = json.loads((BASE_DIR / "results" / "config_congelada.json").read_text())
WESAD_DIR = BASE_DIR / "data" / "WESAD"            # carpeta con S2/, S3/, ... cada una con S*.pkl
FS = 125
VENTANA_S, PASO_S = CFG["ventana_s"], CFG["paso_s"]
RPM_MIN, RPM_MAX = CFG["rpm_min"], CFG["rpm_max"]
FS_CHEST, FS_BVP = 700, 64
CONDICION = {1: "línea base", 2: "estrés", 3: "diversión", 4: "meditación"}
SUJETOS = sorted((p.name for p in WESAD_DIR.glob("S*") if (p / f"{p.name}.pkl").exists()),
                 key=lambda s: int(s[1:]))
print(len(SUJETOS), "sujetos:", SUJETOS)


In [ ]:
# w-2 — Funciones
def leer_sujeto(s):
    with open(WESAD_DIR / s / f"{s}.pkl", "rb") as f:
        d = pickle.load(f, encoding="latin1")
    resp = np.asarray(d["signal"]["chest"]["Resp"], float).ravel()
    bvp = np.asarray(d["signal"]["wrist"]["BVP"], float).ravel()
    lab = np.asarray(d["label"]).ravel()
    return resp, bvp, lab

def marcas_respiracion(resp, fs=FS_CHEST):
    """Inicios de inspiración en la respiración del pecho: picos de la señal filtrada 0.05–1 Hz,
    separados al menos 1 s y con prominencia >= 25 % del rango intercuartílico local."""
    x = decimate(decimate(resp, 7, zero_phase=True), 4, zero_phase=True)   # 700 -> 25 Hz
    f = fs / 28
    b, a = butter(2, [0.05, 1.0], btype="band", fs=f)
    x = filtfilt(b, a, x)
    iqr = np.subtract(*np.percentile(x, [75, 25]))
    picos, _ = find_peaks(x, distance=int(1.0 * f), prominence=0.25 * iqr)
    return picos / f

def rr_ref(m, t0, t1, min_marcas=3):
    m = m[(m >= t0) & (m < t1)]
    return np.nan if len(m) < min_marcas else 60.0 / np.median(np.diff(m))

def rr_fft(seg, fs=FS, nfft=2048, ancho_rpm=2.0):
    q = int(fs // 5)
    x = decimate(decimate(seg, 5, zero_phase=True), q // 5, zero_phase=True)
    x = detrend(x) * np.hanning(len(x))
    f = np.fft.rfftfreq(nfft, q / fs) * 60
    P = np.abs(np.fft.rfft(x, nfft)) ** 2
    banda = (f >= RPM_MIN) & (f <= RPM_MAX)
    k = np.argmax(np.where(banda, P, -np.inf))
    return f[k], P[banda & (np.abs(f - f[k]) <= ancho_rpm)].sum() / P[banda].sum()

def evaluar_sujeto(s):
    resp, bvp, lab = leer_sujeto(s)
    marcas = marcas_respiracion(resp)
    ppg = resample_poly(bvp, FS, FS_BVP)                     # 64 -> 125 Hz, igual que BIDMC
    b, a = butter(4, CFG["lp_fft_hz"], btype="low", fs=FS)
    riiv_fft = filtfilt(b, a, ppg)
    b, a = butter(4, CFG["lp_art_hz"], btype="low", fs=FS)
    riiv = filtfilt(b, a, ppg)
    n5 = 5 * FS
    tipico = np.median([np.std(riiv[i:i + n5]) for i in range(0, len(riiv) - n5 + 1, n5)])
    dur = min(len(ppg) / FS, len(lab) / FS_CHEST)
    filas = []
    for t0 in np.arange(0, dur - VENTANA_S + 1e-9, PASO_S):
        l = lab[int(t0 * FS_CHEST):int((t0 + VENTANA_S) * FS_CHEST)]
        if len(l) == 0 or l.min() != l.max() or l[0] not in CONDICION:
            continue                                         # solo ventanas con una única condición
        i0 = int(round(t0 * FS))
        seg = riiv_fft[i0:i0 + VENTANA_S * FS]
        if len(seg) < VENTANA_S * FS:
            continue
        rr, q = rr_fft(seg)
        exc = max(np.std(riiv[i:i + n5]) for i in range(i0, i0 + VENTANA_S * FS - n5 + 1, n5)) / tipico
        filas.append({"Sujeto": s, "Inicio (s)": t0, "Condición": CONDICION[int(l[0])],
                      "RR pecho (rpm)": rr_ref(marcas, t0, t0 + VENTANA_S),
                      "RR PPG muñeca (rpm)": rr, "Calidad FFT": q,
                      "Artefacto": exc > CFG["umbral_art"]})
    return pd.DataFrame(filas)


In [ ]:
# w-3 — Evaluación en todos los sujetos
W = pd.concat([evaluar_sujeto(s) for s in SUJETOS], ignore_index=True)
W["Válida"] = ~W["Artefacto"] & W["RR pecho (rpm)"].between(RPM_MIN, RPM_MAX)
W["Error"] = (W["RR PPG muñeca (rpm)"] - W["RR pecho (rpm)"]).abs()
W.to_csv(BASE_DIR / "results" / "wesad_rr_muneca_vs_pecho.csv", index=False)
print(f"{len(W)} ventanas; válidas {W['Válida'].sum()} ({100 * W['Válida'].mean():.1f} %)")
print("Ventanas por condición:\n", W.groupby("Condición")["Válida"].agg(["size", "sum"]))


In [ ]:
# w-4 — Resultados: por condición y por sujeto
V = W[W["Válida"]]

def resumen(g):
    por_suj = g.groupby("Sujeto")["Error"].mean()
    return pd.Series({"Sujetos": por_suj.size, "Ventanas": len(g),
                      "MAE mediano por sujeto": por_suj.median(),
                      "MAE medio": g["Error"].mean(),
                      "% |error| ≤ 2 rpm": 100 * (g["Error"] <= 2).mean(),
                      "% |error| ≤ 3 rpm": 100 * (g["Error"] <= 3).mean(),
                      "RR pecho media": g["RR pecho (rpm)"].mean(),
                      "RR muñeca media": g["RR PPG muñeca (rpm)"].mean()})

tabla = pd.concat({"Todas": resumen(V),
                   **{c: resumen(g) for c, g in V.groupby("Condición")}}, axis=1).T.round(2)
print(tabla.to_string())

# Referencia de comparación: predecir la RR media del propio sujeto en línea base
lb = V[V["Condición"] == "línea base"].groupby("Sujeto")["RR pecho (rpm)"].mean()
e_trivial = (V["Sujeto"].map(lb) - V["RR pecho (rpm)"]).abs()
print(f"\nLínea base trivial (RR media del sujeto en reposo): MAE medio {e_trivial.mean():.2f} rpm")
print("\nPor sujeto:")
print(V.groupby("Sujeto")["Error"].agg(["count", "mean", "median"]).round(2).to_string())
